# 04 — Frontier-Vergleich (Phase 5)

Eure Hand-Annotation aus Phase 2 (`annotation/meine_gold.csv`) gegen Frontier-LLM-Annotation derselben 12 Anzeigen (`annotation/frontier_gold.csv`). Output: κ-Tabelle, drei Disagreement-Beispiele, Material fürs Make-or-Buy-Memo (`memo_make_or_buy.md` im Repo-Root).

Cheatsheet: `CHEATSHEETS/frontier-llm-workflow.md`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Frontier-Modell | _ (z. B. `claude-opus-4-7`) |
| Prompt-Variante | _ (z. B. *v1 mit 3 Few-Shots aus IDs A, B, C*) |
| Anzahl Korrektur-Turns | _ |
| Schema-Verletzungs-Mapping | _ (Werte, die ihr gemappt habt — z. B. *möglich → teilweise*) |
| Auffälligkeiten | _ |
| Frontier-CSV | `annotation/frontier_gold.csv` |
| Eigene Gold-CSV | `annotation/meine_gold.csv` |

## Frontier-Daten laden + Schema-Konformität prüfen

In [2]:
import json
import re
from pathlib import Path
from textwrap import dedent

import pandas as pd

KORPUS_PATH       = Path("../daten/eigener_korpus.jsonl")
GOLD_PATH         = Path("../annotation/meine_gold.csv")
FRONTIER_CSV_PATH = Path("../annotation/frontier_gold.csv")
FRONTIER_JSON_TMP = Path("frontier_predictions.json")   # JSON-Array vom Web-Chat zwischenparken

gold_df  = pd.read_csv(GOLD_PATH)
gold_ids = gold_df["id"].astype(str).tolist()
korpus   = pd.read_json(KORPUS_PATH, lines=True)
anzeigen = korpus.set_index("refnr").loc[gold_ids]

FEW_SHOT_IDS = ["10000-1203863577-S", "11949-17214039-S", "13151-1568687-1-S"]

def _str(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = str(v).strip()
    return None if s.lower() in ("", "null", "nan") else s

def _int(v):
    s = _str(v)
    try:
        return int(float(s)) if s is not None else None
    except (ValueError, TypeError):
        return None

def _skills(v):
    s = _str(v)
    if s is None or s.lower() == "nicht_genannt":
        return []
    return [x.strip() for x in s.split("|") if x.strip() and x.strip().lower() != "nicht_genannt"][:3]

def gold_zu_json(row, with_id=False):
    out = {
        "homeoffice":      _str(row.get("homeoffice")),
        "vertragsart":     _str(row.get("vertragsart")),
        "erfahrungslevel": _str(row.get("erfahrungslevel")),
        "gehalt_min_eur":  _int(row.get("gehalt_min_eur")),
        "gehalt_zeitraum": _str(row.get("gehalt_zeitraum")),
        "skills_top3":     _skills(row.get("skills_top3")),
    }
    if with_id:
        out = {"id": row["id"], **out}
    return out

gold_lookup = {str(r["id"]): dict(r) for _, r in gold_df.iterrows()}

# ── Eingabe-Block für claude.ai bauen ─────────────────────────────────────
header = dedent("""\
    Du bist Annotator für deutsche Stellenanzeigen. Annotiere jede Anzeige strikt nach folgendem Schema.
    Antworte ausschließlich mit einem JSON-Array, kein Text davor oder danach, kein Markdown-Codefence.

    SCHEMA (alle 7 Felder pro Anzeige zwingend, inkl. "id"):
    - "id": die Anzeige-ID exakt wie in === id: ... === oben
    - "homeoffice":      "ja" | "teilweise" | "nein" | "remote" | "nicht_genannt"
    - "vertragsart":     "ausbildung" | "festanstellung" | "praktikum" | "werkstudent" | "sonstiges"
    - "erfahrungslevel": "junior" | "mid" | "senior" | "egal" | "nicht_genannt"
    - "gehalt_min_eur":  ganze Zahl (Untergrenze, ohne Tausender-Trennung) ODER null
    - "gehalt_zeitraum": "monat" | "jahr" | null   (null nur, wenn gehalt_min_eur null)
    - "skills_top3":     Array mit max. 3 technischen Skills (oder leeres Array)

    DEFINITIONEN (Auszug aus Schema):
    - "remote"        = 100 % ortsunabhängig
    - "teilweise"     = hybrid / X Tage / mobiles Arbeiten / "Möglichkeit zum hybriden Arbeiten"
    - "ja"            = "Homeoffice" ohne Modalität
    - "nicht_genannt" = nur wenn Anzeige nichts dazu sagt
    - "junior"        = ≤ 2 Jahre Erfahrung; Ausbildung IMMER junior
    - "senior"        = ≥ 5 Jahre / "Senior"-Titel
    - skills_top3:    KEINE Soft Skills, KEINE Sprachen, KEINE Fachgebiete — nur Tools/Sprachen/Frameworks
    - Range "ab 50.000 €" → 50000 als Untergrenze
""")

# 3 Few-Shot-Paare
beispiele = "BEISPIELE:\n\n"
for fs_id in FEW_SHOT_IDS:
    fs_text = korpus.set_index("refnr").loc[fs_id, "text"][:1500]
    fs_json = gold_zu_json(gold_lookup[fs_id], with_id=True)
    beispiele += f"=== id: {fs_id} ===\n{fs_text}\n\nErwartete Annotation:\n{json.dumps(fs_json, ensure_ascii=False)}\n\n"

aufgabe = dedent("""\
    AUFGABE:
    Annotiere die folgenden 12 Anzeigen nach dem Schema oben. Antwort: ein JSON-Array
    mit 12 Objekten, in derselben Reihenfolge wie unten. Jedes Objekt enthält "id"
    plus die 6 Schema-Felder.

    ANZEIGEN:

""")

anzeigen_block = ""
for refnr, row in anzeigen.iterrows():
    anzeigen_block += f"=== id: {refnr} ===\n{row['text']}\n\n"

EINGABE_BLOCK = header + "\n" + beispiele + aufgabe + anzeigen_block

print(f"Eingabe-Block-Länge: {len(EINGABE_BLOCK):,} Zeichen (~{len(EINGABE_BLOCK)//4:,} Tokens)")
print(f"Anzeigen: {len(anzeigen)}, Few-Shots: {len(FEW_SHOT_IDS)}")
print("\nNächste Schritte:")
print("  1. Inhalt von frontier_input_block.txt in claude.ai (Opus 4.6+) oder ChatGPT eingeben.")
print(f"  2. JSON-Array-Antwort in {FRONTIER_JSON_TMP} speichern.")
print("  3. Nächste Zelle ausführen → konvertiert JSON → frontier_gold.csv.")

Path("frontier_input_block.txt").write_text(EINGABE_BLOCK, encoding="utf-8")
print("\nfrontier_input_block.txt geschrieben (öffnen + Inhalt kopieren).")

Eingabe-Block-Länge: 42,086 Zeichen (~10,521 Tokens)
Anzeigen: 12, Few-Shots: 3

Nächste Schritte:
  1. Inhalt von frontier_input_block.txt in claude.ai (Opus 4.6+) oder ChatGPT eingeben.
  2. JSON-Array-Antwort in frontier_predictions.json speichern.
  3. Nächste Zelle ausführen → konvertiert JSON → frontier_gold.csv.

frontier_input_block.txt geschrieben (öffnen + Inhalt kopieren).


In [3]:
# JSON-Array vom Frontier-Chat → annotation/frontier_gold.csv
# Schritt: frontier_predictions.json muss vorher mit dem Output des Chats befüllt sein.

import csv

raw = FRONTIER_JSON_TMP.read_text(encoding="utf-8").strip()
# Falls das Modell doch Begleittext geschrieben hat: erstes JSON-Array herauspulen
match = re.search(r"\[.*\]", raw, re.DOTALL)
if not match:
    raise ValueError(f"Kein JSON-Array in {FRONTIER_JSON_TMP} gefunden — Output prüfen.")
frontier_records = json.loads(match.group(0))
print(f"Frontier-Records geladen: {len(frontier_records)} (erwartet: 12)")

# Schema-Verletzungs-Mapping (häufige Werte, die das Frontier-LLM erfindet)
SCHEMA_VERLETZUNG_MAP = {
    "homeoffice": {
        "möglich": "teilweise", "moeglich": "teilweise",
        "nach absprache": "teilweise", "flexibel": "teilweise",
        "kein homeoffice": "nein", "vollzeit remote": "remote",
        "100% remote": "remote", "vollständig remote": "remote",
    },
    "vertragsart": {
        "freelance": "sonstiges", "selbstaendig": "sonstiges",
        "leiharbeit": "sonstiges", "trainee": "sonstiges",
    },
    "erfahrungslevel": {
        "berufseinsteiger": "junior", "entry": "junior", "anfänger": "junior",
        "mittel": "mid", "expert": "senior", "experte": "senior",
        "alle level": "egal", "egal welches level": "egal",
    },
    "gehalt_zeitraum": {
        "month": "monat", "year": "jahr", "monatlich": "monat", "jährlich": "jahr",
    },
}

def _map(feld, val):
    if val is None:
        return None
    if isinstance(val, (list, int, float)):
        return val
    s = str(val).strip().lower()
    return SCHEMA_VERLETZUNG_MAP.get(feld, {}).get(s, s)

# Records normalisieren + CSV schreiben
FELDER_CSV = ["id", "homeoffice", "vertragsart", "erfahrungslevel",
              "gehalt_min_eur", "gehalt_zeitraum", "skills_top3", "notiz"]

mapped_count = 0
csv_rows = []
for rec in frontier_records:
    rid = rec.get("id")
    row = {"id": rid, "notiz": ""}
    for feld in ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_zeitraum"]:
        original = rec.get(feld)
        mapped   = _map(feld, original)
        if isinstance(original, str) and original.strip().lower() != str(mapped).lower():
            mapped_count += 1
        row[feld] = mapped if mapped is not None else ""
    g = rec.get("gehalt_min_eur")
    row["gehalt_min_eur"] = "" if g is None else int(g) if not isinstance(g, str) else g
    sk = rec.get("skills_top3") or []
    row["skills_top3"] = "|".join(sk) if isinstance(sk, list) else str(sk)
    csv_rows.append(row)

with FRONTIER_CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FELDER_CSV)
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"frontier_gold.csv geschrieben → {FRONTIER_CSV_PATH}")
print(f"Schema-Verletzungs-Mappings angewendet: {mapped_count}")
print("\nSchema-Konformitätscheck im Terminal:")
print("  python annotation/validate.py annotation/frontier_gold.csv")
print("\nKappa Mensch ↔ Frontier (für nächste Zelle, oder direkt im Terminal):")
print("  python annotation/validate.py annotation/meine_gold.csv --kappa-against annotation/frontier_gold.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'frontier_predictions.json'

In [ ]:
from collections import Counter

# ── Hypothese vor κ-Compute ───────────────────────────────────────────────
# Schätzung: Gesamt-κ erwarte ich um 0.50–0.70 (moderat–substanziell).
# Niedrigstes κ erwarte ich bei `homeoffice` (dieselbe Schema-Ambivalenz, die zwischen Menschen
# nur κ=0.12 erzielt hat). Höchstes κ bei `vertragsart` (eindeutig aus API + Text).

# ── κ-Compute (gleicher Algorithmus wie validate.py) ──────────────────────
def cohen_kappa(labels_a, labels_b):
    assert len(labels_a) == len(labels_b)
    n = len(labels_a)
    if n == 0:
        return float("nan")
    agreed = sum(1 for a, b in zip(labels_a, labels_b) if a == b)
    p_o = agreed / n
    cat_a, cat_b = Counter(labels_a), Counter(labels_b)
    p_e = sum((cat_a[c] / n) * (cat_b[c] / n) for c in set(cat_a) | set(cat_b))
    if p_e == 1.0:
        return float("nan")
    return (p_o - p_e) / (1 - p_e)

frontier_df = pd.read_csv(FRONTIER_CSV_PATH).rename(columns={"id": "refnr"})
mein_df     = gold_df.rename(columns={"id": "refnr"})

mein_by_id     = {str(r["refnr"]): dict(r) for _, r in mein_df.iterrows()}
frontier_by_id = {str(r["refnr"]): dict(r) for _, r in frontier_df.iterrows()}
common_ids = sorted(set(mein_by_id) & set(frontier_by_id))

CAT_FELDER = ["homeoffice", "vertragsart", "erfahrungslevel"]

rows = []
disagreements = []   # für die Disagreement-Zelle
for feld in CAT_FELDER:
    a = [str(mein_by_id[r].get(feld) or "").strip() for r in common_ids]
    b = [str(frontier_by_id[r].get(feld) or "").strip() for r in common_ids]
    k = cohen_kappa(a, b)
    agree = sum(1 for x, y in zip(a, b) if x == y)
    rows.append({"feld": feld, "κ": round(k, 3), "Übereinst.": f"{agree/len(common_ids):.0%}"})
    for rid, x, y in zip(common_ids, a, b):
        if x != y:
            disagreements.append({"refnr": rid, "feld": feld, "ich": x, "frontier": y})

kappa_df = pd.DataFrame(rows)
print(f"κ Mensch ↔ Frontier (n={len(common_ids)}):\n")
print(kappa_df.to_string(index=False))

print("\nInterpretation (Landis & Koch): < 0.40 mäßig · 0.41–0.60 moderat · 0.61–0.80 substanziell · > 0.80 fast perfekt")
print(f"\nDisagreements gesamt (kategoriale Felder): {len(disagreements)}")
print("\nAlle Disagreements pro Feld:")
for feld in CAT_FELDER:
    sub = [d for d in disagreements if d["feld"] == feld]
    if not sub:
        continue
    print(f"\n── {feld} ({len(sub)}) ──")
    for d in sub:
        print(f"  {d['refnr']}: ich={d['ich']!r:20s} frontier={d['frontier']!r}")

### Drei konkrete Disagreement-Beispiele (nach dem Run ausfüllen)

Pro Beispiel: refnr, dein Wert, Frontier-Wert, **wer hatte recht** mit Bezug auf Schema-Definition oder Anzeigen-Text.

---

**Disagreement 1 — refnr: _XXXXX_** (Feld: `_`)

- Ich: `_wert_`
- Frontier: `_wert_`
- **Wer hat recht?** _Bezug auf SCHEMA.md-Definition + konkrete Textstelle der Anzeige. Drei Lesarten:_
  - _„Frontier hat recht, Schema sagt eindeutig X — ich habe Schema falsch interpretiert."_
  - _„Ich habe recht, Frontier hat halluziniert / Wert nicht im Text gestützt."_
  - _„Beide nicht eindeutig — Schema-Lücke (Edge Case), den das Schema offen lässt."_

---

**Disagreement 2 — refnr: _XXXXX_** (Feld: `_`)

- Ich: `_`
- Frontier: `_`
- **Wer hat recht?** _

---

**Disagreement 3 — refnr: _XXXXX_** (Feld: `_`)

- Ich: `_`
- Frontier: `_`
- **Wer hat recht?** _

---

### Material fürs Memo (`memo_make_or_buy.md` im Repo-Root)

Aus der κ-Tabelle + den drei Disagreements:
- Auf welchen Feldern ist Frontier vertrauenswürdig (κ ≥ 0.6)?
- Auf welchen Feldern systematisch unsicher (κ < 0.4)?
- Wie viele der Disagreements waren „Frontier-Fehler" vs. „Schema-Lücke"?
- Würde ich die restlichen 78 Anzeigen vom Frontier annotieren lassen, selbst, oder hybrid (Frontier zuerst + Mensch reviewt schwache Felder)?